<a href="https://colab.research.google.com/github/HojuneLee0106/CodeTree/blob/main/personal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers transformers torch numpy
!pip install -q google-genai
!pip install -q chromadb
!pip install -q fastapi uvicorn pyngrok
!pip install -q "ragas==0.2.10" "langchain-community>=0.2,<0.4" "langchain-google-genai" "datasets"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 128.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 121.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is th

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from torchvision import models
import math

In [3]:
from datasets import load_dataset
import re

ds = load_dataset("heegyu/open-korean-instructions", split="train")

pairs = []
for row in ds:
    text = row["text"]
    usr = re.findall(r"<usr>(.*?)(?=<sys>|<bot>|<usr>|$)", text, re.DOTALL)
    bot = re.findall(r"<bot>(.*?)(?=<sys>|<bot>|<usr>|$)", text, re.DOTALL)
    if usr and bot:
        q, a = usr[0].strip(), bot[0].strip()
        if q and a:
            pairs.append((q, a))

print(f"Q&A 쌍: {len(pairs):,}개")

# SPM 학습에 챗봇 표현도 포함시키기 위해 chat.txt로 저장
with open("chat.txt", "w", encoding="utf-8") as f:
    for q, a in pairs:
        f.write(f"질문: {q}\n답변: {a}\n\n")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/3.15k [00:00<?, ?B/s]

OIG-smallchip2-ko.json:   0%|          | 0.00/106M [00:00<?, ?B/s]

koalpaca.json:   0%|          | 0.00/38.9M [00:00<?, ?B/s]

korquad-chat.json:   0%|          | 0.00/23.4M [00:00<?, ?B/s]

sharegpt_deepl_ko.json:   0%|          | 0.00/611M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/375159 [00:00<?, ? examples/s]

Q&A 쌍: 375,078개


In [4]:


import sentencepiece as spm

# 말뭉치로 직접 vocab 학습
spm.SentencePieceTrainer.train(
    input="chat.txt",
    model_prefix="ko_spm",
    vocab_size=8000,
    character_coverage=0.9995,
    model_type="bpe",
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
    input_sentence_size=500_000,   # 최대 50만 문장만 샘플링 (핵심)
    shuffle_input_sentence=True,   # 랜덤 샘플링
    num_threads=4,                 # 병렬 처리
)

sp = spm.SentencePieceProcessor()
sp.load("ko_spm.model")

# 기존 encode/decode 교체
vocab_size = sp.get_piece_size()   # 8000
encode     = lambda s: sp.encode(s, out_type=int)
decode     = lambda ids: sp.decode(ids)

print(sp.encode("서울의 수도는 어디인가요?", out_type=str))
# ['▁서울의', '▁수도는', '▁어디', '인가요', '?']

['▁서', '울', '의', '▁수도', '는', '▁어디', '인가요', '?']


In [5]:
block_size   = 256
n_embd       = 384
n_head       = 6
n_layer      = 6
batch_size   = 256
steps        = 10_000
lr_max       = 3e-4
lr_min       = 3e-5
warmup_steps = 400
dropout      = 0.1
grad_clip    = 1.0

In [6]:
import random
def build_qa_samples():
    bos, eos = sp.bos_id(), sp.eos_id()
    samples = []
    for q, a in pairs:
        q_ids = encode(f"질문: {q}\n답변:")
        a_ids = encode(f" {a}") + [eos]
        ids = [bos] + q_ids + a_ids
        if len(ids) > block_size + 1:
            continue
        x = ids[:-1]
        y = ids[1:]
        y = [-100] * len(q_ids) + y[len(q_ids):]   # 질문 구간 마스킹
        samples.append((x, y))
    return samples

all_samples = build_qa_samples()
random.shuffle(all_samples)                        # 섞은 뒤 분리
n_val = int(0.1 * len(all_samples))
val_samples   = all_samples[:n_val]
train_samples = all_samples[n_val:]

print(f"train 샘플: {len(train_samples):,}개  val 샘플: {len(val_samples):,}개")

train 샘플: 275,014개  val 샘플: 30,557개


In [7]:
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(device)

cuda


In [8]:
class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv  = nn.Linear(n_embd, 3*n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd,    bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        hd      = C // n_head
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        def sh(t): return t.view(B, T, n_head, hd).transpose(1, 2)
        q, k, v = sh(q), sh(k), sh(v)
        att  = q @ k.transpose(-2, -1) / hd**0.5
        mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device))
        att  = att.masked_fill(~mask, float("-inf"))
        att  = self.drop(F.softmax(att, dim=-1))
        out  = (att @ v).transpose(1, 2).reshape(B, T, C)
        return self.drop(self.proj(out))

In [9]:
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1  = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention()
        self.ln2  = nn.LayerNorm(n_embd)
        self.mlp  = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd, bias=False), nn.GELU(),
            nn.Linear(4*n_embd, n_embd, bias=False), nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

In [10]:
import torch.nn.functional as F
class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks  = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.head    = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        T = idx.size(1)
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        x = self.ln_f(self.blocks(x))
        logits = self.head(x)
        loss = None
        if targets is not None:
            # targets에서 -100인 위치(질문 부분)는 loss 계산 제외
            loss = F.cross_entropy(
                logits.view(-1, vocab_size),
                targets.view(-1),
                ignore_index=-100   # ← 이게 핵심 변경점
            )
        return logits, loss

    #generate수정버전
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=40,
                repetition_penalty=1.3, stop_token=None):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            logits = logits[:, -1, :] / temperature

            # 반복 패널티 (부호 고려: 양수는 나누고 음수는 곱함)
            for tok in set(idx[0].tolist()):
                if logits[0, tok] > 0:
                    logits[0, tok] /= repetition_penalty
                else:
                    logits[0, tok] *= repetition_penalty

            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float("-inf")

            probs = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, 1)
            idx = torch.cat([idx, next_tok], dim=1)
            if stop_token and next_tok.item() == stop_token:
                break
        return idx

In [11]:
@torch.no_grad()
def estimate_loss(model, eval_iters=20):
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [12]:
def get_lr(step):
    if step < warmup_steps:
        return lr_max * step / warmup_steps
    t = (step - warmup_steps) / max(1, steps - warmup_steps)
    return lr_min + (lr_max - lr_min) * 0.5 * (1 + math.cos(math.pi * t))

In [13]:
def train(model):
    opt = torch.optim.AdamW(model.parameters(), lr=lr_max, weight_decay=0.01)

    def lr_at(step):
        if step < warmup_steps:
            return lr_max * (step + 1) / warmup_steps
        ratio = (step - warmup_steps) / max(1, steps - warmup_steps)
        return lr_min + 0.5 * (lr_max - lr_min) * (1 + math.cos(math.pi * ratio))

    model.train()
    for step in range(steps):
        for g in opt.param_groups:
            g["lr"] = lr_at(step)

        xb, yb = get_batch("train")
        logits, loss = model(xb, yb)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        opt.step()

        if step % 1000 == 0 or step == steps - 1:
            losses = estimate_loss(model)
            print(f"step {step:5d}  lr {opt.param_groups[0]['lr']:.1e}  "
                  f"train {losses['train']:.3f}  val {losses['val']:.3f}")

    torch.save(model.state_dict(), "mini_gpt_qa.pt")
    print("QA 학습 완료 → mini_gpt_qa.pt 저장")

In [18]:
def chat(model):
    model.load_state_dict(torch.load("mini_gpt_qa.pt", map_location=device))
    model.eval()
    print("Type a prompt and the model will continue it (empty line to quit).")

    bos = sp.bos_id()
    while True:
        try:
            prompt = input("> ")
        except EOFError:
            break
        if not prompt:
            break

        # 학습 때와 똑같은 형식으로 감싸기
        q_ids = encode(f"질문: {prompt}\n답변:")
        ids = [bos] + q_ids
        if not q_ids:
            print("(no tokens from the prompt are in the vocabulary)")
            continue

        idx = torch.tensor([ids], dtype=torch.long, device=device)
        out = model.generate(idx, 200, stop_token=sp.eos_id())[0].tolist()
        print(decode(out[len(ids):]))   # 입력(bos+질문) 제외, 생성 부분만 출력

In [15]:
import random
def get_batch(split: str = "train"):
    pool = train_samples if split == "train" else val_samples
    PAD = sp.pad_id()
    batch = random.sample(pool, batch_size)
    maxlen = max(len(x) for x, _ in batch)
    xb, yb = [], []
    for x, y in batch:
        pad = maxlen - len(x)
        xb.append(x + [PAD] * pad)
        yb.append(y + [-100] * pad)
    xb = torch.tensor(xb, dtype=torch.long, device=device)
    yb = torch.tensor(yb, dtype=torch.long, device=device)
    return xb, yb

In [16]:
model = MiniGPT().to(device)
train(model)

step     0  lr 7.5e-07  train 9.148  val 9.148
step  1000  lr 3.0e-04  train 4.447  val 4.473
step  2000  lr 2.8e-04  train 3.768  val 3.833
step  3000  lr 2.5e-04  train 3.363  val 3.489
step  4000  lr 2.2e-04  train 3.156  val 3.279
step  5000  lr 1.7e-04  train 3.019  val 3.157
step  6000  lr 1.3e-04  train 2.919  val 3.087
step  7000  lr 9.0e-05  train 2.850  val 3.043
step  8000  lr 5.8e-05  train 2.836  val 3.012
step  9000  lr 3.7e-05  train 2.784  val 2.969
step  9999  lr 3.0e-05  train 2.779  val 2.984
QA 학습 완료 → mini_gpt_qa.pt 저장


In [19]:
chat(model)

Type a prompt and the model will continue it (empty line to quit).
> 안녕?
안녕하세요! 오늘은 무엇을 도와드릴까요?
> 너 누구야?
드워프는 18세기의 나이키 선수입니다.
> 안녕하세요
죄송하지만 무슨 뜻인지 잘 모르겠습니다. 자세한 정보를 제공하신가요?
> 반가워!
다음은 `a`와 같은 각 요소를 기준으로 단일 튜플로 변환하는 코드입니다. a = b.append(a)) #경고: 이 코드 생성은 실험적입니다. 실행하기 전에 버그가 있는지 코드를 검사하십시오.
> 반가워
안녕하세요! 오늘은 무엇을 도와드릴까요?
> 스트레스 완화 팁 알려줘
마음챙김을 유지하고 불안의 원인을 파악해 주세요. 자신을 돌보고 감정을 표현할 수 있는 일을 하십시오. 또한 충분한 수면을 취하고 규칙적으로 운동하여 긴장을 풀고 스트레스를 받을 시간을 가지십시오.
> 미국의 총기 소유에 관한 법률은 무엇입니까?
미국 헌법과 법안이 있습니다. 연방부, 유엔 및 기타 정부 간섭의 일부입니다.
> 
